In [3]:
!pip install torch
!pip install torch-geometric
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root='data/Cora', name='Cora')
data = dataset[0]

print(data)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 32.8 MB/s eta 0:00:0000:0100:01


Processing...


Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


Done!


In [4]:
import networkx as nx

G = nx.Graph()
G.add_edges_from(data.edge_index.t().tolist())

In [6]:
print(G)

Graph with 2708 nodes and 5278 edges


In [7]:
def clustering(G):
    return nx.average_clustering(G)

In [8]:
def run_null_models(G, N=50):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    degree_seq = [d for _, d in G.degree()]
    
    results = {
        "ER": [],
        "Configuration": [],
        "BA": [],
        "SmallWorld": [],
    }
    
    for _ in range(N):

        # Erdős–Rényi (preserves expected density only)
        p = (2*m) / (n*(n-1))
        ER = nx.erdos_renyi_graph(n, p)
        results["ER"].append(clustering(ER))

        # Configuration Model (preserves degree sequence exactly)
        CM = nx.configuration_model(degree_seq)
        CM = nx.Graph(CM)        # remove parallel edges
        CM.remove_edges_from(nx.selfloop_edges(CM))
        results["Configuration"].append(clustering(CM))

        # Preferential Attachment (Barabási–Albert)
        k = int(np.mean(degree_seq) / 2)  # approx matches edge count
        BA = nx.barabasi_albert_graph(n, k)
        results["BA"].append(clustering(BA))

        # Small-World (Watts–Strogatz)
        k_ws = int(np.mean(degree_seq))   # similar mean degree
        SW = nx.watts_strogatz_graph(n, k_ws, p=0.1)
        results["SmallWorld"].append(clustering(SW))

    return results


In [10]:
import numpy as np
results = run_null_models(G, N=50)
real_clustering = clustering(G)


In [11]:
import pandas as pd

summary = pd.DataFrame({
    "Model": list(results.keys()),
    "Mean Clustering": [np.mean(results[m]) for m in results],
    "Std Dev": [np.std(results[m]) for m in results],
})

print("Real CORA clustering:", real_clustering)
summary

Real CORA clustering: 0.24067329850193736


,Model,Mean Clustering,Std Dev
0,ER,0.001237,0.000440
1,Configuration,0.006301,0.001111
2,BA,0.000000,0.000000
3,SmallWorld,0.000081,0.000290
